In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Abe-Suzuki Earthquake Network — Japan JMA Catalog (1985-2025)

Run as a script  : python JAPAN_network_ABE.py
Convert to notebook: python convert_to_notebook.py JAPAN_network_ABE.py notebooks/JAPAN_network_ABE.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import seaborn as sns

from src.network import build_abe_suzuki_network, discretize_space_3d
from src.metrics import (
    estimate_gamma_mle,
    test_power_law,
    verify_balanced_degrees,
    measure_preferential_attachment,
    plot_preferential_attachment,
)
from src.assortativity import (
    attach_catalog_attrs,
    compute_assortativity,
    plot_assortativity,
    plot_knn,
    plot_directed_mixing,
    plot_rich_club,
)
from src.robustness import simulate_robustness, plot_robustness_curves
from src.centrality import (
    compute_all_centralities,
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
    compute_bb_fitness,
    plot_bb_fitness,
    plot_bb_fitness_geo,
    plot_bb_fitness_theory,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_directed_louvain,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
    analyze_weak_ties,
    plot_weak_ties,
    analyze_condensation,
    plot_condensation_geo,
)
from src.nullmodels import build_null_graphs, compare_metrics, plot_degree_comparison
from src.signed import (
    build_signed_network,
    plot_signed_degree,
    compute_structural_balance,
    analyze_chains,
    plot_chains,
    plot_signed_geo_map,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    analyze_in_out_degree_distribution,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/Japan")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)
CUT_YEAR    = 1985
TARGET_CRS  = "epsg:32654"  # UTM Zone 54N — appropriate for central Japan
SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "japan" / "abe")

_JP_BOUNDS = dict(west=122, east=154, south=22, north=48)
MAP_WIDTH  = 845
MAP_HEIGHT = 780

## Data Loading

In [ ]:
print("Loading Japan earthquake catalog (JMA via ISC)...")
df = pd.read_csv(DATA_DIR / "japan_earthquakes_jma_1985_2025_m1_5.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

neg = df_net[df_net["depth_km"] < 0]
print(f"\nNegative-depth events: {len(neg)} — kept, collapse to cell_z = -1")

## Network Construction

In [ ]:
G_5km  = build_abe_suzuki_network(df_net, cell_size_km=5,  target_crs=TARGET_CRS)
G_10km = build_abe_suzuki_network(df_net, cell_size_km=10, target_crs=TARGET_CRS)

print(f"\n5 km network:  {G_5km.number_of_nodes():,} nodes  {G_5km.number_of_edges():,} edges")
print(f"10 km network: {G_10km.number_of_nodes():,} nodes  {G_10km.number_of_edges():,} edges")

## Hub Map — 2D

The top-2% highest-degree cells are mapped geographically to locate the
most seismically active spatial clusters. Degree here is total weighted
degree (sum of transition counts), so hubs correspond to cells that were
visited most frequently in the earthquake sequence — the busiest fault segments.

Expected pattern for Japan: hubs should cluster along the **Japan Trench**
(off NE Honshu), the **Sagami Trough** (Kanto region), and the **Median
Tectonic Line** fault system. The Tohoku megathrust zone (around 38–40°N,
142–144°E) should produce the highest-degree nodes given the 2011 M9.0 sequence.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "cell_id": n,
        "degree":  G_10km.degree(n),
        "lat":     G_10km.nodes[n]["lat"],
        "lon":     G_10km.nodes[n]["lon"],
    }
    for n in G_10km.nodes() if "lat" in G_10km.nodes[n]
])
df_nodes  = df_nodes[df_nodes["degree"] > 0]
threshold = df_nodes["degree"].quantile(0.98)
df_hubs   = df_nodes[df_nodes["degree"] >= threshold].copy()
print(f"Mapping top 2% hubs: {len(df_hubs)} cells (degree >= {threshold:.0f})")

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="degree", size="degree",
    color_continuous_scale="inferno",
    map_style="carto-positron",
    hover_name="cell_id",
    title="Abe-Suzuki Network Hubs: Top 2% Most Active Seismic Cells — Japan (10x10x10 km)",
)
fig.update_layout(margin={"r": 0, "t": 40, "l": 0, "b": 0}, width=MAP_WIDTH, height=MAP_HEIGHT,
                  map=dict(center={"lat": 36.0, "lon": 138.0}, zoom=0, bounds=_JP_BOUNDS))
save_plotly(fig, "hub_map_2d_japan_10km")
fig.show()

## Hub Map — 3D

Three-dimensional scatter (lon, lat, depth) with the Japan coastline
projected at $z = 0$ km. This view disentangles shallow crustal hubs
(upper crust, 0–30 km), intermediate slab hubs (30–100 km), and deep
slab hubs (> 300 km in northern Honshu/Hokkaido), making it possible to
assess whether the most active network nodes correspond to specific
tectonic regimes: megathrust interface, intraplate crustal, or deep slab.

In [ ]:
url      = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world    = gpd.read_file(url)
japan_geom  = world[world["ADMIN"] == "Japan"].geometry.iloc[0]
geometries  = ([japan_geom] if japan_geom.geom_type == "Polygon"
               else list(japan_geom.geoms))

df_hubs["depth_km"] = df_hubs["cell_id"].apply(lambda x: float(x.split("_")[2]) * 10)

lons_map, lats_map, zs_map = [], [], []
for polygon in geometries:
    cx, cy = polygon.exterior.coords.xy
    lons_map.extend(cx); lats_map.extend(cy); zs_map.extend([0] * len(cx))
    lons_map.append(None); lats_map.append(None); zs_map.append(None)

fig_3d = px.scatter_3d(
    df_hubs, x="lon", y="lat", z="depth_km",
    color="degree", size="degree", color_continuous_scale="inferno",
    hover_name="cell_id",
    hover_data={"lat": ":.2f", "lon": ":.2f", "depth_km": True, "degree": True},
)
fig_3d.add_trace(go.Scatter3d(
    x=lons_map, y=lats_map, z=zs_map,
    mode="lines", line=dict(color="blue", width=3),
    name="Japan Surface (0 km)", hoverinfo="skip",
))
fig_3d.update_layout(
    template="plotly_white",
    scene=dict(
        zaxis=dict(autorange="reversed", title="Depth (km)"),
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        aspectratio=dict(x=1, y=1.5, z=0.6),
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    title="3D View of Seismic Hubs with Japan Coastline",
)
fig_3d.update_traces(marker=dict(line=dict(width=0)))
fig_3d.show()

## Degree Balance Verification

By construction, every interior node in the Abe-Suzuki network must satisfy
$k_i^{\text{in}} = k_i^{\text{out}}$ (the cell that received the $n$-th event
also sent the $(n+1)$-th). Only the first and last events in the catalog
produce a single unbalanced node each.

A non-empty list here signals a data-loading or sorting error — the catalog
must be strictly ordered by time before network construction.

In [ ]:
print("Checking degree balance (interior nodes: in-strength == out-strength)...")
result = verify_balanced_degrees(G_5km)
if isinstance(result, list):
    print(f"Unbalanced nodes: {result}")

## Degree Distribution — Linear Scale

Linear-scale histogram of node degrees, truncated at $k = 50$ to reveal the
low-degree bulk of the distribution. In a scale-free network this range is
dominated by nodes with very few connections; the exponential-like drop-off
here contrasts with the heavy tail visible on the log-log plots.

In [ ]:
plot_degree_distribution_linear(G_5km,  "Abe-Suzuki Japan (5 km)",  max_degree=50)
plot_degree_distribution_linear(G_10km, "Abe-Suzuki Japan (10 km)", max_degree=50)

## Degree Distribution — Log-Log Scatter

Log-log scatter of $P(k)$ vs $k$ is the canonical diagnostic for a power-law
tail: a straight line with slope $-\gamma$ on this plot implies $P(k) \propto k^{-\gamma}$.
Abe & Suzuki (2004) found $\gamma \approx 2.2$ for the Japan JMA catalog —
the original benchmark. Our replication with 40 years of JMA data (1985–2025)
tests whether this exponent is stable over time and across different magnitude thresholds.

In [ ]:
analyze_in_out_degree_distribution(G_5km,  "Abe-Suzuki Japan (5 km)")
analyze_in_out_degree_distribution(G_10km, "Abe-Suzuki Japan (10 km)")

analyze_degree_distribution(G_5km,  "Abe-Suzuki Japan (5x5x5 km)")
analyze_degree_distribution(G_10km, "Abe-Suzuki Japan (10x10x10 km)")

## Degree Distribution — Log Binning

Logarithmic binning reduces noise in the sparse high-$k$ tail by grouping
degrees into exponentially spaced bins and normalising by bin width.
This is the standard method recommended by Clauset et al. (2009) for
visually assessing power-law fit quality; the slope of the fitted line
provides a visual estimate of $\gamma$ before the more rigorous MLE.

In [ ]:
analyze_degree_distribution_log_binning(G_5km,  "Abe-Suzuki Japan (5x5x5 km)",   k_min_fit=10)
analyze_degree_distribution_log_binning(G_10km, "Abe-Suzuki Japan (10x10x10 km)", k_min_fit=10)

## Degree Distribution — CCDF

The complementary cumulative distribution function $P(K \geq k)$ avoids
binning artefacts entirely and is monotone non-increasing by construction.
For a power-law $P(k) \propto k^{-\gamma}$, the CCDF follows
$P(K \geq k) \propto k^{-(\gamma-1)}$; a straight line on a log-log CCDF
plot with slope $-(\gamma-1)$ is therefore the cleanest visual confirmation.

In [ ]:
plot_ccdf_with_fit(G_5km,  "Abe-Suzuki Japan 5 km",  k_min_fit=10)
plot_ccdf_with_fit(G_10km, "Abe-Suzuki Japan 10 km", k_min_fit=10)

## MLE Gamma Estimation

The maximum-likelihood estimator for the power-law exponent (Clauset et al. 2009):
$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$
where $n$ is the number of degrees $\geq k_{\min}$ and $k_{\min} = 10$.

The Clauset-Shalizi-Newman (CSN) likelihood-ratio test compares the power-law
to an exponential alternative: $R > 0$ with $p < 0.05$ rejects the exponential
and supports the power-law. Abe & Suzuki (2004) found $\gamma \approx 2.2$
for the Japan catalog — values $\gamma < 3$ imply a divergent second moment and
enhanced vulnerability to targeted attack. Direct comparison with the original
result is the primary replication test of this section.

In [ ]:
gamma_10km = float("nan")
for label, G in [("5 km", G_5km), ("10 km", G_10km)]:
    degs  = [d for _, d in G.degree(weight="weight") if d > 0]
    gamma = estimate_gamma_mle(degs, k_min=10)
    print(f"MLE γ (Japan {label} network): {gamma:.3f}")
    if label == "10 km":
        gamma_10km = gamma

    result = test_power_law(degs, k_min=10)
    print(f"  CSN test: γ={result['gamma']} (σ={result['sigma']})  "
          f"R={result['R']}  p={result['p_value']}  → {result['verdict']}")

## Preferential Attachment

## Preferential Attachment

In the Barabási-Albert (BA) model, new edges attach to existing nodes with
probability proportional to their degree — a *rich-get-richer* mechanism that
generates power-law degree distributions.  The attachment kernel is

$$\pi(k) \propto k^{\alpha},$$

where $\alpha = 1$ (linear) recovers standard BA and $\alpha \neq 1$ indicates
a more complex growth rule.

The empirical estimator (Jeong *et al.* 2003) replays the time-ordered edge
sequence.  At each step $t$, both the source cell $c_t$ and target cell
$c_{t+1}$ receive a degree increment $\Delta k = 1$.  Binning all
(node, pre-increment degree) pairs and computing the mean increment per
degree class gives:

$$\pi(k) = \frac{\sum_{i:\, k_i(t)=k} \Delta k_i}{\#\{i:\, k_i(t)=k\}}$$

Japan is particularly interesting for this test: the 2011 Tohoku sequence
generated an extreme aftershock cascade, suddenly elevating the degree of the
offshore Miyagi cells by orders of magnitude.  If $\alpha > 1$ (super-linear),
this burst feeding further activity would be detectable.

**Reference:** Jeong H., Néda Z. & Barabási A.-L. (2003). Measuring
preferential attachment in evolving networks. *Europhysics Letters* 61, 567–572.

In [ ]:
print("Measuring preferential attachment kernel π(k) (10 km network)...")
ks_pa, pi_pa, alpha_pa = measure_preferential_attachment(
    df_net, cell_size_km=10.0, target_crs=TARGET_CRS,
)
print(f"  Attachment exponent α = {alpha_pa:.3f}")
plot_preferential_attachment(ks_pa, pi_pa, alpha_pa,
                             title="Abe-Suzuki Japan (10 km)")

## Macroscopic Metrics

The small-world signature (Watts & Strogatz 1998) requires simultaneously
$C_{\text{real}} \gg C_{\text{ER}}$ (high local clustering) and
$L_{\text{real}} \approx L_{\text{ER}}$ (short average path length),
where $C_{\text{ER}} \approx \langle k \rangle / N$ and $L_{\text{ER}} \approx \ln N / \ln \langle k \rangle$.

**Giant component fraction** measures network integrity: values $> 0.9$ indicate
a well-connected seismic system. Japan's high seismicity rate across many
tectonic regimes (subduction, collision, intraplate) may produce a larger and
more densely connected giant component than Italy's predominantly thrust system.

In [ ]:
G = G_10km

print("--- Giant Component ---")
wcc     = list(nx.weakly_connected_components(G))
G_giant = G.subgraph(max(wcc, key=len)).copy()
pct     = G_giant.number_of_nodes() / G.number_of_nodes() * 100
print(f"Components: {len(wcc)} | Giant: {G_giant.number_of_nodes():,} nodes ({pct:.1f}%)")

print("\n--- Clustering Coefficient ---")
G_und = G_giant.to_undirected()
G_und.remove_edges_from(nx.selfloop_edges(G_und))
avg_c = nx.average_clustering(G_und)
p_rnd = (2 * G_und.number_of_edges()
         / (G_giant.number_of_nodes() * (G_giant.number_of_nodes() - 1)))
print(f"Avg clustering C = {avg_c:.4f}  (Erdos-Renyi expected: {p_rnd:.4f})")

print("\n--- Average Path Length & Diameter ---")
t0       = time.time()
avg_L    = nx.average_shortest_path_length(G_und)
diameter = nx.diameter(G_und)
print(f"L = {avg_L:.2f}  |  Diameter = {diameter}  ({time.time()-t0:.0f}s)")

print("\n--- Adjacency Matrix Sparsity ---")
adj      = nx.to_scipy_sparse_array(G)
sparsity = 1.0 - adj.nnz / G.number_of_nodes()**2
print(f"Shape: {adj.shape}  |  Non-zeros: {adj.nnz:,}  |  Sparsity: {sparsity*100:.4f}%")

plt.figure(figsize=(8, 8))
plt.spy(adj, markersize=0.05, color="darkblue")
plt.title("Adjacency Matrix (10 km Japan network)", fontsize=14)
plt.xlabel("Target Cell Index", fontsize=12)
plt.ylabel("Source Cell Index", fontsize=12)
plt.legend(handles=[
    mpatches.Patch(facecolor="darkblue", label="Link (edge present)"),
    mpatches.Patch(facecolor="white", edgecolor="lightgray", label="No link"),
], loc="upper right", fontsize=9, framealpha=0.8)
plt.tight_layout()
savefig("adjacency_matrix_japan_10km")
plt.show()

## Centrality Analysis

Eight complementary centrality measures capture distinct structural roles:
**Degree** (raw activity), **PageRank** (stress-sink importance),
**Closeness** (topological proximity to all others),
**Betweenness** (fault-bridge role on shortest paths),
**Eigenvector** (rich-club membership), **Katz** (all-path influence),
**HITS Hub** (cells that trigger many others), **HITS Authority** (destination cells).

Seismological expectation: hubs should cluster along the Japan Trench interface
and the Median Tectonic Line; betweenness-central nodes should connect
geographically separated tectonic provinces (e.g., Tohoku and Kanto regions).

In [ ]:
print("Computing all 8 centrality measures (10 km network)...")
df_centrality = compute_all_centralities(G_10km, k_betweenness=1000, seed=42)

for metric, label in [
    ("Degree",      "Most Active Swarms"),
    ("PageRank",    "Stress Sinks"),
    ("Closeness",   "Topological Centers"),
    ("Betweenness", "Fault Bridges"),
    ("Eigenvector", "Rich-Club Cores"),
    ("Katz",        "All-Path Influential"),
    ("HITS_Hub",    "Seismic Triggers"),
    ("HITS_Auth",   "Seismic Destinations"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "Abe-Suzuki Japan (10 km)")
plot_top_n_cells(df_centrality, top_n=10, title="Abe-Suzuki Japan (10 km)")

## Centrality Geo Maps

Geographic projection of the top-10 nodes per centrality metric onto the
Japan basemap. The **convergence map** (colour = count of metrics for which a
node ranks in the top-10) identifies structurally robust hubs: cells that are
simultaneously active, well-connected, and located on critical fault bridges.

Seismological expectation: hubs should concentrate offshore (Japan Trench)
for degree and PageRank, while betweenness-central nodes may appear onshore
at transition zones between tectonic regimes — seismic bridges between the
Pacific subduction system and the Philippine Sea collision zone.

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_JP_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_JP_BOUNDS,
)

## Bianconi-Barabasi Fitness

## Bianconi–Barabási Fitness and Bose-Einstein Condensation

Standard preferential attachment (Barabási & Albert 1999) produces
scale-free networks by a *first-mover advantage*: early nodes grow faster
simply because they have had more time to accumulate edges.  The
**Bianconi–Barabási** (BB) model (2001) extends this by assigning each node
an intrinsic **fitness** $\eta_i$, with attachment probability

$$\pi_i \propto \eta_i\, k_i.$$

The degree of cell $i$, born at time $t_i$, grows as

$$k_i(t) \approx m \left(\frac{t}{t_i}\right)^{\beta_i}, \qquad \beta_i = \frac{\eta_i}{C},$$

and the empirical fitness estimator is

$$\hat{\beta}_i = \frac{\ln k_i(T)}{\ln(T / t_i)}.$$

**Seismological interpretation.** For Japan, the 2011 Tohoku sequence
dramatically elevated the degree of offshore Miyagi cells.  If these cells
show anomalously high $\hat{\beta}$, it suggests Bose-Einstein condensation —
a single megathrust zone capturing a finite fraction of all seismic activity.
The Lorenz curve and Gini coefficient quantify how unequal the degree distribution
is when sorted by fitness, separating seniority bias from intrinsic productivity.

### References
Bianconi G. & Barabási A.-L. (2001). Competition and multiscaling in
evolving networks. *Europhysics Letters* 54, 436–442.

Bianconi G. & Barabási A.-L. (2001). Bose-Einstein condensation in
complex networks. *Physical Review Letters* 86, 5632–5635.

Barabási A.-L. & Albert R. (1999). Emergence of scaling in random networks.
*Science* 286, 509–512.

In [ ]:
print("Computing Bianconi-Barabasi fitness (10 km network)...")
df_bb = compute_bb_fitness(G_10km, df_net, cell_size_km=10.0, target_crs=TARGET_CRS)
print(f"  {len(df_bb)} cells with valid fitness estimates.")
display(df_bb.sort_values("fitness_beta", ascending=False).head(10))
plot_bb_fitness(df_bb, title="Abe-Suzuki Japan (10 km)")
plot_bb_fitness_theory(df_bb, gamma=gamma_10km, title="Abe-Suzuki Japan (10 km)")
plot_bb_fitness_geo(
    df_bb, title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0,
    height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_JP_BOUNDS,
)

## Community Detection — Full Suite

Modularity maximisation: $Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i,c_j)$
(Newman & Girvan 2004), where $m$ is edge count and $\delta$ is the Kronecker delta.
Four algorithms — **Louvain**, **Consensus Louvain** (100-run majority vote),
**Spectral** (Laplacian eigenvectors), and **InfoMap** (minimum description length)
— are run and their pairwise NMI computed.

Seismological expectation: communities should correspond to geographically
contiguous tectonic provinces — NE Japan arc (Pacific subduction), SW Japan
arc (Philippine Sea subduction), Ryukyu arc, and possibly the Izu-Bonin arc.
High NMI across methods would confirm that community structure is robust
rather than algorithm-dependent.

In [ ]:
print("Building undirected giant component for community detection...")
G_und_giant = G_giant.to_undirected()
G_und_giant.remove_edges_from(nx.selfloop_edges(G_und_giant))

# ── Louvain (single run, seed=42) ────────────────────────────────────────────
print("Louvain...")
community_map = run_louvain(G_und_giant, seed=42)
k_louvain = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G_und_giant, community_map,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_JP_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

# ── Consensus Louvain (100 runs) ─────────────────────────────────────────────
print("Consensus Louvain (100 runs)...")
community_map_consensus = run_consensus_louvain(G_und_giant, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G_und_giant, community_map_consensus,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_JP_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

# ── Spectral clustering (k = Louvain count) ───────────────────────────────────
print(f"Spectral clustering (k={k_louvain})...")
community_map_spectral = run_spectral(G_und_giant, k=k_louvain, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities")
plot_community_geo(
    G_und_giant, community_map_spectral,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_JP_BOUNDS,
    min_community_size=50, method_name="Spectral",
)

# ── InfoMap (undirected flow) ─────────────────────────────────────────────────
print("InfoMap...")
community_map_infomap = run_infomap(G_und_giant, directed=False, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G_und_giant, community_map_infomap,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_JP_BOUNDS,
    min_community_size=50, method_name="InfoMap",
)

# ── NMI comparison heatmap ────────────────────────────────────────────────────
print("Computing NMI across methods...")
partitions = {
    "Louvain":   community_map,
    "Consensus": community_map_consensus,
    "Spectral":  community_map_spectral,
    "InfoMap":   community_map_infomap,
}
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="Abe-Suzuki Japan (10 km)")

## Adjacency Matrix — Community Order

Reordering nodes by Louvain community reveals the **block structure** hidden in the
raw adjacency matrix. Within each community nodes are sorted by degree descending,
so hub nodes appear at the leading edge of each block.

The block-density heatmap shows how strongly each pair of tectonic provinces
are coupled seismically — high off-diagonal density would indicate inter-arc
stress transfer, a key indicator of regional seismic hazard propagation.

In [ ]:
import scipy.sparse as sp

_comm_ids_sorted = sorted(set(community_map.values()))
_node_order = sorted(
    G_und_giant.nodes(),
    key=lambda n: (_comm_ids_sorted.index(community_map.get(n, 0)),
                   -G_und_giant.degree(n))
)
_N = len(_node_order)
_n2i = {n: i for i, n in enumerate(_node_order)}

_r, _c = [], []
for u, v in G_und_giant.edges():
    if u in _n2i and v in _n2i:
        _r += [_n2i[u], _n2i[v]]; _c += [_n2i[v], _n2i[u]]
_A = sp.csr_matrix((np.ones(len(_r), dtype=np.float32), (_r, _c)), shape=(_N, _N))

_bs = max(1, _N // 800)
_nb = _N // _bs
_Ads = np.zeros((_nb, _nb), dtype=np.float32)
for _bi in range(_nb):
    _Ads[_bi] = np.asarray(
        _A[_bi * _bs: (_bi + 1) * _bs].sum(axis=0)
    ).ravel()[: _nb * _bs].reshape(_nb, _bs).sum(axis=1)

_comm_sizes = [sum(1 for n in _node_order if community_map.get(n) == c)
               for c in _comm_ids_sorted]
_boundaries = np.cumsum(_comm_sizes)[:-1] / _bs

_nc = len(_comm_ids_sorted)
_ci = {c: i for i, c in enumerate(_comm_ids_sorted)}
_B = np.zeros((_nc, _nc))
for u, v in G_und_giant.edges():
    i = _ci.get(community_map.get(u, -1), 0)
    j = _ci.get(community_map.get(v, -1), 0)
    _B[i, j] += 1
    if i != j:
        _B[j, i] += 1
for i in range(_nc):
    for j in range(_nc):
        s_i, s_j = _comm_sizes[i], _comm_sizes[j]
        denom = s_i * s_j if i != j else s_i * (s_i - 1) / 2
        if denom > 0:
            _B[i, j] /= denom

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
ax.imshow(np.log1p(_Ads), aspect="auto", cmap="Blues", interpolation="nearest")
for bnd in _boundaries:
    ax.axhline(bnd, color="white", lw=0.6, alpha=0.8)
    ax.axvline(bnd, color="white", lw=0.6, alpha=0.8)
ax.set_title("Reordered Adjacency Matrix\n(nodes sorted by community, then by degree)",
             fontsize=12)
ax.set_xlabel("Target Cell (community order)", fontsize=10)
ax.set_ylabel("Source Cell (community order)", fontsize=10)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.legend(handles=[
    mpatches.Patch(facecolor=plt.cm.Blues(0.85), label="Links (darker = more)"),
    mpatches.Patch(facecolor="white", edgecolor="lightgray", label="No links"),
], loc="lower right", fontsize=8, framealpha=0.8)

im = axes[1].imshow(np.log1p(_B * 1000), cmap="YlOrRd", aspect="auto")
axes[1].set_title(f"Community Block-Density Matrix\n({_nc} Louvain communities)",
                  fontsize=12)
axes[1].set_xlabel("Community j", fontsize=10)
axes[1].set_ylabel("Community i", fontsize=10)
plt.colorbar(im, ax=axes[1], label="log(1 + density × 1000)")

fig.suptitle("Adjacency Structure by Community — Japan 10 km", fontsize=14, y=1.01)
plt.tight_layout()
savefig("adjacency_matrix_community_order_japan")
plt.show()

## Directed Community Detection

Directed modularity $Q_d = \frac{1}{m}\sum_{ij}\left[A_{ij} - \frac{k_i^{\text{out}}k_j^{\text{in}}}{m}\right]\delta(c_i,c_j)$
(Leicht & Newman 2008) groups nodes that preferentially send *and* receive
seismic activity within the same community.

In Japan's asymmetric subduction setting, directed communities may reveal
one-way stress cascades: the megathrust consistently triggers inland crustal
events but is rarely triggered in return — a directional asymmetry that
undirected modularity cannot capture.

In [ ]:
print("Running directed Louvain (Leiden algorithm) on G_10km...")
community_map_directed = run_directed_louvain(G_10km, seed=42)
k_directed = len(set(community_map_directed.values()))
print(f"  {k_directed} directed communities  (vs {k_louvain} undirected)")

plot_community_geo(
    G_10km, community_map_directed,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_JP_BOUNDS,
    min_community_size=50, method_name="Directed Louvain (Leiden)",
)

partitions_full = {**partitions, "Directed Louvain": community_map_directed}
nmi_full = compute_nmi_matrix(partitions_full)
display(nmi_full.round(3))
plot_nmi_heatmap(nmi_full, title="Abe-Suzuki Japan — directed vs undirected")

## Community Markov Flow

Once the Louvain partition assigns each cell to community $c \in \{1,\dots,K\}$,
the directed weighted network is coarse-grained into a $K \times K$ flow matrix:

$$F_{ij} = \sum_{\substack{u \in C_i \\ v \in C_j}} w_{uv}$$

Row-normalisation yields the **Markov transition matrix**

$$T_{ij} = \frac{F_{ij}}{\sum_k F_{ik}}$$

so that $T_{ij}$ is the probability that seismic activity in community $i$
moves to community $j$ in the next event. This is the community-level analogue
of the random-walk transition used by Rosvall & Bergstrom (2008) in the InfoMap
framework, applied here to coarse-grained tectonic provinces rather than individual cells.

Three quantities characterise each state $i$:

**Self-retention** $T_{ii}$ — fraction of flow that stays inside community $i$.
**Shannon entropy** $H_i = -\sum_j T_{ij} \log_2 T_{ij}$ (bits).
**Stationary distribution** $\boldsymbol{\pi}$ satisfying $\boldsymbol{\pi} = \boldsymbol{\pi} T$.

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008). Maps of random walks on complex
networks reveal community structure. *PNAS*, 105(4), 1118–1123.

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

print("Building community flow matrix (Louvain, 10 km)...")
flow_count_df = build_community_flow_matrix(G_10km, community_map)
T_markov = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
print(f"Dominant attractor:   Community {df_flow.iloc[0]['community']} "
      f"(π = {df_flow.iloc[0]['stationary']:.4f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="Abe-Suzuki Japan (10 km)")
plot_flow_entropy(df_flow, K=K_comm, title="Abe-Suzuki Japan (10 km)")
plot_stationary_distribution(df_flow, title="Abe-Suzuki Japan (10 km)")

out_path = RESULTS_DIR / "data" / "japan_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Granovetter Weak Ties

Granovetter (1973) "The Strength of Weak Ties": edges connecting distant
social groups are typically low-weight (rare contacts), while strong ties
cluster within groups. The **bridge fraction** per weight bin measures the
proportion of edges in that bin that cross Louvain community boundaries.

Seismological interpretation: in Japan, low-weight inter-arc transitions
(rare stress transfer from NE Japan arc to SW Japan arc or Ryukyu arc)
should have a higher bridge fraction than frequent within-arc transitions.
If confirmed, the small-world structure of the Japan seismic network is
maintained by rare but topologically critical cross-arc stress jumps.

In [ ]:
print("Analysing weak ties...")
df_wt = analyze_weak_ties(G_und_giant, community_map, n_bins=8)
display(df_wt)
plot_weak_ties(df_wt, title="Abe-Suzuki Japan (10 km)")

## Condensation Graph — SCC Analysis

A strongly connected component (SCC) is a maximal set of nodes with directed
paths between all pairs. The condensation DAG classifies each SCC as:
**source** (no incoming inter-SCC edges), **sink** (no outgoing), or **transit**.

In seismic terms: source SCCs are regions that consistently *initiate* sequences
without being triggered by others — candidate mainshock nucleation zones.
Sink SCCs absorb seismic activity as terminal aftershock clusters.
For Japan, we expect the Japan Trench interface cells to appear in large
source SCCs (megathrust as primary driver) and inner Honshu cells as sinks.

In [ ]:
print("Computing condensation graph...")
C_japan, df_scc_japan = analyze_condensation(G_10km, min_scc_size=2)
print(f"  SCCs: {C_japan.number_of_nodes()}  "
      f"(source={(df_scc_japan['role']=='source').sum()}  "
      f"sink={(df_scc_japan['role']=='sink').sum()}  "
      f"transit={(df_scc_japan['role']=='transit').sum()})")
display(
    df_scc_japan[df_scc_japan["size"] >= 5]
    .sort_values("size", ascending=False)
    .head(20)
)
plot_condensation_geo(
    df_scc_japan, min_size=5,
    title="Abe-Suzuki Japan (10 km)",
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_JP_BOUNDS,
)

## Assortativity Analysis

Newman (2003) assortativity coefficient:
$r = \frac{\sum_{ij}(A_{ij} - k_i k_j/2m)\,x_i x_j}{\sum_{ij}(k_i\delta_{ij} - k_i k_j/2m)\,x_i^2}$
where $x_i$ is a node attribute (degree, depth, magnitude mean).

**Degree assortativity** $r < 0$ (disassortative) is the canonical scale-free
signature. **Depth assortativity** for Japan is of particular interest: the
three-layer depth structure (shallow crustal / slab interface / deep slab)
predicts a positive depth assortativity if tectonic layers generate internally
clustered sequences, or near-zero if cross-depth triggering is common.

In [ ]:
print("Attaching catalog attributes to nodes...")
attach_catalog_attrs(G_10km, df_net, cell_size_km=10.0, target_crs=TARGET_CRS)

print("Computing assortativity measures...")
df_assort = compute_assortativity(G_10km)
display(df_assort)

print("\nMixing pattern plots:")
plot_assortativity(G_10km, title="Abe-Suzuki Japan (10 km)")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G_10km, title="Abe-Suzuki Japan (10 km)", gamma=gamma_10km)

print("Directed degree mixing (in/out):")
plot_directed_mixing(G_10km, title="Abe-Suzuki Japan (10 km)")

print("Rich-club coefficient:")
plot_rich_club(G_10km, title="Abe-Suzuki Japan (10 km)")

## Robustness Analysis

Albert et al. (2000) showed that scale-free networks exhibit a sharp
dichotomy: **robust** to random node failure but **fragile** to targeted
attack on hubs. The critical fraction $f_c$ (at which GCC $< 0.05$) under
targeted attack is the key metric.

For Japan, $f_c$ may be lower than Italy or the US if the Tohoku sequence
created extreme-degree hubs that, when removed, rapidly fragment the network.
The Japan network's robustness directly quantifies how dependent its global
connectivity is on a small number of megathrust cells.

In [ ]:
n_gcc = G_und_giant.number_of_nodes()
m_gcc = G_und_giant.number_of_edges()
G_er_rob = nx.gnm_random_graph(n_gcc, m_gcc, seed=42)

print("Simulating robustness...")
rob_results = {
    "Random (Real)":   simulate_robustness(G_und_giant, strategy="random",   seed=42),
    "Targeted (Real)": simulate_robustness(G_und_giant, strategy="targeted"),
    "Random (ER)":     simulate_robustness(G_er_rob,    strategy="random",   seed=42),
    "Targeted (ER)":   simulate_robustness(G_er_rob,    strategy="targeted"),
}
plot_robustness_curves(rob_results, title="Abe-Suzuki Japan (10 km)")

## Null Model Comparison

Five synthetic benchmarks isolate which structural properties are explained
by degree sequence alone vs additional spatial or community constraints:
- **ER** (Erdős-Rényi): random baseline with Poisson $P(k)$
- **BA** (Barabási-Albert): preferential attachment, power-law $P(k)$
- **WS** (Watts-Strogatz): rewired ring lattice with small-world properties
- **SBM**: stochastic block model fitted from Louvain communities
- **Config**: configuration model (Molloy & Reed 1995) — preserves exact degree sequence

If the Japan network's $\gamma \approx 2.2$ (Abe & Suzuki 2004) is confirmed,
its degree distribution should match BA better than ER and differ from Config
in clustering, confirming non-trivial spatial organisation beyond the degree sequence.

In [ ]:
print("Building null models (ER, BA, WS, SBM, Config)...")
null_graphs = build_null_graphs(G_und_giant, community_map=community_map,
                                include_config=True, seed=42)

print("Overlaying degree distributions...")
plot_degree_comparison(
    G_und_giant, null_graphs,
    title="Abe-Suzuki Japan (10 km)",
    k_min_fit=10,
)

print("Computing structural metrics (path length is approximate for large graphs)...")
df_null = compare_metrics(G_und_giant, null_graphs, n_path_samples=150, seed=42)
display(df_null)

## Spatial Interaction Null Model

The ER/BA/WS/SBM/config null models test whether the network's topology is
surprising *relative to its degree sequence*. The gravity model provides a
geographic control (Wilson 1971):

$$\lambda_{ij} = C \cdot A_i^{\,\alpha} \cdot A_j^{\,\beta} \cdot d_{ij}^{-\eta}$$

where $A_i$ is out-strength, $A_j$ is in-strength, and $d_{ij}$ is haversine distance.
The **distance decay exponent** $\eta$ is the key parameter: for seismic networks
the expected value is lower than for human mobility because fault rupture can
propagate stress over hundreds of kilometres. Japan's megathrust geometry predicts
very low $\eta$ — long-distance offshore-to-onshore triggering dominates.

*References:* Wilson, A. G. (1971). Environment and Planning A, 3(1), 1–32.
Krings, G. et al. (2009). J. Stat. Mech., L07003.

In [ ]:
from src.spatial_nulls import fit_gravity_model, plot_gravity_fit, plot_distance_decay, plot_excess_map

print("Fitting gravity null model on Japan 10 km network...")
gravity_params, df_gravity = fit_gravity_model(G_10km, min_distance_km=1.0)
print(f"Gravity model (α={gravity_params['alpha']}, β={gravity_params['beta']}, "
      f"η={gravity_params['eta']}, R²={gravity_params['r2']})")
print(f"  Fitted on {gravity_params['n_edges']:,} edges "
      f"({gravity_params['n_vertical']:,} vertical transitions excluded)")
print(f"  Median structural excess: {df_gravity['excess'].median():.2f}×")
print(f"  Top-5% excess threshold:  {df_gravity['excess'].quantile(0.95):.1f}×")
display(df_gravity.nlargest(10, "excess")[
    ["u", "v", "w", "d_km", "lambda_", "excess", "lat_u", "lon_u", "lat_v", "lon_v"]
])

plot_gravity_fit(df_gravity, gravity_params, title="Abe-Suzuki Japan (10 km)")
plot_distance_decay(df_gravity, gravity_params, title="Abe-Suzuki Japan (10 km)")
plot_excess_map(
    df_gravity, title="Abe-Suzuki Japan (10 km)", n_top=200,
    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_JP_BOUNDS,
)

out_path = RESULTS_DIR / "data" / "japan_spatial_null.csv"
df_gravity[["u", "v", "w", "d_km", "lambda_", "log_excess", "excess"]].to_csv(out_path, index=False)
print(f"Spatial null results saved → {out_path}")

## Personalized PageRank — Tohoku 2011

Personalized PageRank (PPR) teleports the random walker exclusively to seed $s$:
$$\pi_s(v) = \alpha \sum_u \frac{\pi_s(u)}{d^+(u)} + (1-\alpha)\,\mathbf{1}[v=s]$$
with $\alpha = 0.85$. High-PPR cells are topologically *reachable* from the
Tohoku 2011 mainshock cell (M9.0, 38.322°N 142.369°E), defining the
**network-theoretic aftershock influence zone**.

The 2011 Tohoku earthquake generated the most extensive aftershock zone in
the JMA catalog. Comparing the PPR footprint to the empirically documented
aftershock distribution (Koper et al. 2011) tests whether network topology
predicts real seismic diffusion — the PPR influence zone should extend
along the Japan Trench and inland toward the Kanto region.

In [ ]:
TOHOKU_LAT, TOHOKU_LON = 38.322, 142.369

def _nearest_node(G: nx.DiGraph, lat: float, lon: float) -> str:
    return min(
        (n for n in G.nodes() if "lat" in G.nodes[n]),
        key=lambda n: (G.nodes[n]["lat"] - lat)**2 + (G.nodes[n]["lon"] - lon)**2,
    )

seed_tohoku = _nearest_node(G_10km, TOHOKU_LAT, TOHOKU_LON)
print(f"Seed node for Tohoku 2011: {seed_tohoku}  "
      f"(lat={G_10km.nodes[seed_tohoku]['lat']:.3f}, "
      f"lon={G_10km.nodes[seed_tohoku]['lon']:.3f})")

ppr_tohoku = nx.pagerank(G_10km, alpha=0.85,
                          personalization={seed_tohoku: 1.0},
                          weight="weight")

df_ppr = pd.DataFrame([
    {"cell_id": n, "ppr": ppr_tohoku[n],
     "lat": G_10km.nodes[n]["lat"], "lon": G_10km.nodes[n]["lon"]}
    for n in G_10km.nodes() if "lat" in G_10km.nodes[n]
]).sort_values("ppr", ascending=False)

print("Top 10 cells by PPR from Tohoku seed:")
display(df_ppr.head(10)[["cell_id", "lat", "lon", "ppr"]])

df_ppr_plot = df_ppr.head(500).copy()
df_ppr_plot["ppr_log_size"]  = np.log1p(df_ppr_plot["ppr"] / df_ppr_plot["ppr"].max() * 1e4) + 1
df_ppr_plot["ppr_log_color"] = np.log10(df_ppr_plot["ppr"])
df_ppr_plot = df_ppr_plot.sort_values("ppr_log_color", ascending=True)  # low PPR rendered first → high PPR (yellow) on top

fig_ppr = px.scatter_map(
    df_ppr_plot,
    lat="lat", lon="lon",
    color="ppr_log_color", size="ppr_log_size", size_max=25,
    color_continuous_scale="plasma",
    hover_name="cell_id",
    hover_data={"ppr": ":.6f"},
    map_style="carto-positron",
    title="Personalized PageRank from Tohoku 2011 — Seismic Influence Zone",
)
fig_ppr.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0}, width=MAP_WIDTH, height=MAP_HEIGHT,
                      coloraxis_colorbar=dict(title="log₁₀(PPR)"),
                      map=dict(center={"lat": 38.0, "lon": 142.0}, zoom=0, bounds=_JP_BOUNDS))
save_plotly(fig_ppr, "ppr_tohoku_2011")
fig_ppr.show()

## Force-Directed Layout

The Fruchterman-Reingold spring model places nodes using attractive forces
along edges and repulsive forces between all pairs, ignoring geographic
coordinates entirely. Coloring by Louvain community then by latitude tests
whether **topology faithfully encodes geography** (Fortunato 2010).

If community color aligns with latitude band → topological communities
reproduce the north–south arc zonation of the Japan subduction system.
If communities are mixed geographically → non-geographic (long-range) stress
couplings exist, consistent with megathrust sequences that trigger activity
across multiple arc segments simultaneously.

In [ ]:
LAYOUT_N = 400
top_nodes = sorted(G_und_giant.nodes(), key=lambda n: G_und_giant.degree(n), reverse=True)[:LAYOUT_N]
G_sub = G_und_giant.subgraph(top_nodes).copy()

print(f"Running spring layout on top-{LAYOUT_N} nodes by degree...")
pos = nx.spring_layout(G_sub, seed=42, k=0.3)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

n_comms = len(set(community_map.values()))
comm_cmap = plt.colormaps["tab20"].resampled(n_comms)
node_colors_comm = [comm_cmap(community_map.get(n, 0) % n_comms) for n in G_sub.nodes()]
nx.draw_networkx_nodes(G_sub, pos, ax=axes[0], node_size=30,
                       node_color=node_colors_comm, alpha=0.8)
nx.draw_networkx_edges(G_sub, pos, ax=axes[0], alpha=0.08, width=0.5, edge_color="gray")
axes[0].set_title("Coloured by Louvain Community", fontsize=12)
axes[0].axis("off")

lats = np.array([G_und_giant.nodes[n].get("lat", 0.0) for n in G_sub.nodes()])
lat_norm = (lats - lats.min()) / max(lats.max() - lats.min(), 1e-6)
node_colors_geo = plt.cm.RdYlBu(lat_norm)
nx.draw_networkx_nodes(G_sub, pos, ax=axes[1], node_size=30,
                       node_color=node_colors_geo, alpha=0.8)
nx.draw_networkx_edges(G_sub, pos, ax=axes[1], alpha=0.08, width=0.5, edge_color="gray")
axes[1].set_title("Coloured by Latitude (blue=south, red=north)", fontsize=12)
axes[1].axis("off")

fig.suptitle(f"Force-Directed Layout: Japan Network (top {LAYOUT_N} nodes)\n"
             "If community ≈ latitude → topology reproduces arc geography",
             fontsize=13, y=1.01)
plt.tight_layout()
savefig("force_directed_layout_japan")
plt.show()

## k-core Decomposition

The $k$-core is the maximal subgraph in which every node has degree $\geq k$
(Seidman 1983). Iterative shell peeling reveals a nested hierarchy from
peripheral (low-$k$) to backbone (high-$k$) nodes.

The innermost shell identifies the **seismogenic backbone** of the Japan network:
cells that are densely mutually connected. Given Japan's extreme seismicity rate,
the maximum core number $k_{\max}$ may be substantially higher than Italy or
the US, and the inner core should be concentrated along the Japan Trench
and the Tohoku aftershock zone.

In [ ]:
core_numbers = nx.core_number(G_und_giant)
df_core = pd.DataFrame([
    {"cell_id": n, "core": core_numbers[n],
     "lat": G_und_giant.nodes[n].get("lat", None),
     "lon": G_und_giant.nodes[n].get("lon", None),
     "degree": G_und_giant.degree(n)}
    for n in G_und_giant.nodes()
]).dropna(subset=["lat", "lon"])

print(f"Core number range: {df_core['core'].min()} – {df_core['core'].max()}")
print(f"Inner core (top 10%): {df_core['core'].quantile(0.9):.0f}+")
display(df_core.nlargest(10, "core")[["cell_id", "lat", "lon", "degree", "core"]])

fig_core = px.scatter_map(
    df_core,
    lat="lat", lon="lon",
    color="core", size="core", size_max=18,
    color_continuous_scale="plasma",
    hover_name="cell_id",
    hover_data={"core": True, "degree": True},
    map_style="carto-positron",
    title="k-core Decomposition — Japan (10 km, GCC)",
)
fig_core.update_layout(margin={"r": 0, "t": 50, "l": 0, "b": 0}, width=MAP_WIDTH, height=MAP_HEIGHT,
                       coloraxis_colorbar=dict(title="Core number"),
                       map=dict(center={"lat": 36.0, "lon": 138.0}, zoom=0, bounds=_JP_BOUNDS))
save_plotly(fig_core, "kcore_geo_japan_10km")
fig_core.show()

core_sizes = df_core.groupby("core").size().reset_index(name="n_nodes")
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(core_sizes["core"], core_sizes["n_nodes"],
           color="indigo", alpha=0.85, s=40, zorder=3)
ax.vlines(core_sizes["core"], 0, core_sizes["n_nodes"],
          color="indigo", alpha=0.4, linewidth=1.2)
ax.set_xscale("log")
ax.set_xlabel("Core number k (log scale)", fontsize=12)
ax.set_ylabel("Nodes in k-core", fontsize=12)
ax.set_title("k-core Size Distribution — Japan (10 km)", fontsize=13)
ax.grid(axis="both", ls="--", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
kmax = core_sizes["core"].max()
nmax = core_sizes.loc[core_sizes["core"] == kmax, "n_nodes"].values[0]
ax.annotate(f"k$_{{max}}$={kmax}\n({nmax} nodes)",
            xy=(kmax, nmax), xytext=(kmax * 0.55, nmax * 1.4),
            fontsize=9, color="indigo",
            arrowprops=dict(arrowstyle="->", color="indigo", lw=0.8))
plt.tight_layout()
savefig("kcore_size_distribution_japan")
plt.show()

## Signed Network Analysis

Heider (1946) balance theory: a triangle is **balanced** if
$\prod_{\text{edges}} \text{sign} > 0$ (all same sign, or two negative).
Edges are labelled $+1$ if $M_{B} \geq M_{A}$ (escalation) or $-1$ if
$M_{B} < M_{A}$ (decay).

For Japan, the expected balance ratio is near-random (≈ 50%), as confirmed
for Italy and the US. However, the Tohoku sequence may produce local regions
of elevated positive balance (prolonged escalation to the M9.0) followed
by long negative chains (aftershock decay over months), potentially creating
measurable spatial variation in signed structure across the fault system.

In [ ]:
print("Building signed network...")
G_signed = build_signed_network(df_net, target_crs=TARGET_CRS, cell_size_km=10.0)

n_pos = sum(1 for _, _, d in G_signed.edges(data=True) if d["sign"] > 0)
n_neg = sum(1 for _, _, d in G_signed.edges(data=True) if d["sign"] < 0)
print(f"  Positive edges: {n_pos:,}  ({n_pos/(n_pos+n_neg):.1%})")
print(f"  Negative edges: {n_neg:,}  ({n_neg/(n_pos+n_neg):.1%})")

plot_signed_degree(G_signed, title="Abe-Suzuki Japan (10 km)")
plot_signed_geo_map(G_signed, title="Japan (10 km)",
                    center_lat=36.0, center_lon=138.0, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
                    bounds=_JP_BOUNDS)

print("Computing structural balance...")
balance = compute_structural_balance(G_signed)
print(f"  Triangles: {balance['n_triangles']:,}")
print(f"  Balanced:  {balance['n_balanced']:,} ({balance['balance_ratio']:.1%})")
print(f"  Frustration index: {balance['frustration']:.4f}")

print("Analysing magnitude run lengths...")
df_chains = analyze_chains(df_net)
display(df_chains)
plot_chains(df_chains, title="Japan catalog")

## Export Results

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "japan_eq_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved to {out_path}  ({len(df_final):,} rows)")

for fname, G in [("japan_G5km.pkl", G_5km), ("japan_G10km.pkl", G_10km)]:
    pkl_path = RESULTS_DIR / "cache" / fname
    with open(pkl_path, "wb") as f:
        pickle.dump(G, f)
    print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "japan_G10km.gexf"
nx.write_gexf(G_10km, gexf_path)
print(f"Gephi export → {gexf_path}")